WARNING: Notebook was written during development. May not work as intended. Retained here only for completion and future reference.

# Sheathed catheter insertion with a moving vessel and wall contact

The rod is fixed in the lab and the vessel is translated onto it with
`ObstacleMovement`, as in `rod_through_artery`. At depth `d_k` the vessel mouth sits at
`(L − d_k) t₀`, and every node that has not yet entered the vessel is clamped at zero
displacement and rotation, so the part outside the vessel is held straight and cannot
buckle or loop. Only the inserted length is free, and the wall is the only thing
steering it.

- **Why the loop happened before.** With only the base clamped, the span between the
  clamp and the vessel mouth is a long unsupported column. The wall reaction at the
  tip compresses it, and it buckles. A real catheter is held there by the introducer
  sheath, which is what the clamp on the outside nodes models.
- **Penalty.** `k_w` is set so that one element's contact spring `k_w L_e` is a fixed
  fraction `β` of its bending spring `EI / L_e³`, which gives `k_w = β E I / L_e⁴`. This
  is `_contact.compute_self_contact_stiffness`, and the same scaling holds for the
  wall. The penetration under a reaction `F` on one element is
  `δ = F / (k_w L_e) = F L_e³ / (β EI)`, so it falls with the cube of the element
  length. Run once, read the largest `ε / r`, and raise `β` only if it exceeds a few
  percent. A much larger `k_w` makes the contact tangent dwarf the bending tangent, and
  Newton then overshoots in and out of the thin smoothing band, which costs
  iterations or diverges.
- **Units.** SI, so the magnetic loads can be switched on later without rescaling.

### Imports

In [ ]:
import dataclasses
import functools as ft
import pathlib
import time
import typing

import imageio.v2 as imageio
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polyscope as ps
import scipy.interpolate as spy_interp
import trimesh

import crops.src.bc as _bc
import crops.src.contact as _contact
import crops.src.fe_cosserat as _fe
import crops.src.mesher as _mesher
import crops.src.solver as _solver
import crops.src.utils as _utils
import crops.src.viz as _viz
import crops.tools.geom as _geom

jax.config.update("jax_enable_x64", True)
plt.rcParams.update(_viz.high_res_plot_settings)

results_dir = pathlib.Path("crops/data/results/catheter_contact")
results_dir.mkdir(parents=True, exist_ok=True)

### Target, vessel, material, solver, and baseline settings

In [ ]:
target_file = pathlib.Path("crops/data/targets/helix_a3_l20.csv")
stl_file = pathlib.Path("crops/data/obstacles/helix_a3_l20.stl")
length_scale = 1.0  # csv units to m, use 1e-3 if the csv is in mm
stl_length_scale = 1.0  # stl units to m

# Catheter tube with the elastomer of Wang (2025).
outer_radius = 0.5e-3  # [m]
inner_radius = 0.35e-3  # [m]
shear_modulus = 3.03e5  # [Pa]


nonlinear_settings = {
  "max_iter": 100,
  "threshold": 1e-8,
  "abs_threshold": 1e-11,
  "lam_min": 0.01,
  "line_search_max_iter": 12,
  "line_search_shrink": 0.5,
  "line_search_alpha_min": 1e-6,
  "line_search_armijo_c": 1e-4,
}
linear_settings = {"solver": _solver.LinearSolvers.SCIPY_SPARSE, "rtol": 1e-10,}
solver_verbose = True


baseline_settings = {
  "num_elems": 120,
  "num_fibers": 8,
  "steps_per_elem": 2.0,
  "sheath_fraction": 0.1,
  "grid_spacing": 0.2e-3,
  "penalty_ratio": 1.0,
}

### Setup container and builder

In [ ]:
class ContactSetup(typing.NamedTuple):
  """Stores every object one sheathed insertion needs.

  Element and fiber counts, reference coordinates, contact radii, and section
  properties are read from `rod`, and the SDF grid and penalty from
  `problem.obstacle`, rather than duplicated here.

  Attributes:
    rod: `Rod` of the straight catheter from the origin along the entry tangent `t₀`,
      fixed in the lab.
    problem: `CosseratProblem` holding the rod, the clamp schedule of the outside
      nodes, the Newton solver, and the vessel obstacle whose `ObstacleMovement`
      places its mouth at `(L − d_k) t₀` at step `k`.
    insertion_depths: Array of shape `(num_steps,)` [m] of the free length `d_k` at
      every insertion step, rising uniformly to the target path length.
    node_path_coords: Array of shape `(num_steps, num_nodes)` [m] of the path
      coordinate `σ = s − L + d_k` of every node, positive once the node is inside the
      vessel.
    target_points: Array of shape `(num_samples, 3)` [m] of the dense vessel
      centerline in the vessel body frame, used for plotting and the SDF sign check.
    entry_tangent: Array of shape `(3,)` of the unit tangent `t₀` of the centerline at
      the entry, which is also the insertion direction.
    origin: Array of shape `(3,)` [m] of the first csv point in the csv frame. The
      centerline and the vessel are both shifted by it, so the mouth is the body
      frame origin.
  """

  rod: _mesher.Rod
  problem: _fe.CosseratProblem
  insertion_depths: np.ndarray
  node_path_coords: np.ndarray
  target_points: np.ndarray
  entry_tangent: np.ndarray
  origin: np.ndarray


def build_setup(
  num_elems: int,
  num_fibers: int,
  steps_per_elem: float,
  sheath_fraction: float,
  grid_spacing: float,
  penalty_ratio: float,
) -> ContactSetup:
  """Builds the fixed catheter, the moving vessel, and the clamp schedule.

  The rod lies straight from the origin along the entry tangent and stays fixed in the
  lab. The vessel body frame has its mouth at the origin and its entry tangent along
  `t₀`, and at step `k` it is translated by `T_k = (L − d_k) t₀`, which slides it over
  the rod by `d_k`. Every node whose path coordinate is not positive is outside the
  vessel, so all its dofs are clamped at zero. All other nodes are free and are
  steered by the wall alone.

  Algorithm.
    1. Read the centerline, convert it to m, shift its arc coordinate to start at
       zero and its points to start at the origin, and fit a cubic spline `p(σ)`.
       Set the path length `ℓ` and the entry tangent `t₀ = p′(0) / ‖p′(0)‖`.
    2. Set the rod length `L = (1 + sheath_fraction) ℓ`, place `num_fibers` fibers on
       a circle of the mean radius `(r_o + r_i) / 2`, each with area
       `π (r_o² − r_i²) / num_fibers`, and build `r(s) = s t₀` for `s ∈ [0, L]`.
    3. Set `K = ⌈steps_per_elem · num_elems / (1 + sheath_fraction)⌉`, so the wall
       advances by at most `h / steps_per_elem` per step for the element length `h`,
       which keeps the contact state from jumping across several elements in one
       Newton solve. Set `d_k = (k + 1) Δ` with `Δ = ℓ / K`, and clamp all dofs of the
       nodes with `σ_n^k = s_n − L + d_k ≤ 0` at zero.
    4. Load the vessel, convert it to m, and apply the same shift as the centerline.
       Size the SDF grid from `grid_spacing` over the vessel extent plus a padding of
       three contact radii on each side.
    5. Set `k_w = β E I_min / L_e⁴` with `β = penalty_ratio` and `E = 3G`, and build
       the obstacle from the mesh with the movement `T_k = (L − d_k) t₀`.
    6. Assemble the Newton solver and the problem with the obstacle.

  Args:
    num_elems: Number of rod elements, positive. Penetration scales with `L_e³`, so
      refinement is the cheapest way to reduce it at fixed `β`.
    num_fibers: Number of fibers around the tube wall, positive, typically 8 to 16.
    steps_per_elem: Number of insertion steps per element length, at least one and
      typically two.
    sheath_fraction: Length left outside the vessel at full insertion as a fraction
      of the path length, in `(0, 1)`. It must exceed one element length over `ℓ`.
    grid_spacing: SDF grid spacing [m], positive. It should resolve the wall
      thickness with three or more cells and stay below half the contact radius.
    penalty_ratio: Ratio `β` of the element contact spring to the element bending
      spring, typically 0.1 to 2.

  Returns:
    A `ContactSetup` holding every object one insertion needs.
  """
  target = pd.read_csv(target_file)
  target_arc = length_scale * (target["s"].values - target["s"].values[0])
  target_xyz = length_scale * target[["x", "y", "z"]].values
  target_curve_fn = spy_interp.CubicSpline(target_arc, target_xyz - target_xyz[0])

  path_length = float(target_arc[-1])
  entry_tangent = target_curve_fn(0.0, 1)
  entry_tangent = entry_tangent / np.linalg.norm(entry_tangent)
  rod_length = (1.0 + sheath_fraction) * path_length

  mean_radius = 0.5 * (outer_radius + inner_radius)
  fiber_area = np.pi * (outer_radius**2 - inner_radius**2) / num_fibers
  rod = _mesher.build_rod_geometry(
    analytical_curve=lambda arc: arc * jnp.asarray(entry_tangent),
    parametric_coords=jnp.linspace(0.0, rod_length, num_elems + 1),
    fiber_offsets=jnp.asarray(
      _geom.annular_fiber_offsets(
        num_elems=num_elems, num_fibers=num_fibers, semi_axis_d1=mean_radius
      )
    ),
    fiber_areas=jnp.full((num_elems, num_fibers), fiber_area),
    fiber_mass_density=jnp.full((num_elems, num_fibers), 1100.0),
    fiber_centroid_moi=jnp.zeros((num_elems, num_fibers, 2)),
    seed_binormal=jnp.asarray(np.eye(3)[np.argmin(np.abs(entry_tangent))]),
  )

  num_steps = int(np.ceil(steps_per_elem * num_elems / (1.0 + sheath_fraction)))
  insertion_depths = path_length * np.arange(1, num_steps + 1) / num_steps
  node_path_coords = (
    np.asarray(rod.arc_coords)[None, :] - rod_length + insertion_depths[:, None]
  )
  outside_nodes = [np.flatnonzero(coords <= 0.0) for coords in node_path_coords]
  clamped_dofs = (
    _utils.Translation.X,
    _utils.Translation.Y,
    _utils.Translation.Z,
    _utils.Rotation.THETA_X,
    _utils.Rotation.THETA_Y,
    _utils.Rotation.THETA_Z,
  )
  sheath = [
    _bc.DirichletBC(
      values=(dof, [np.zeros(nodes.size) for nodes in outside_nodes]),
      node_targets=outside_nodes,
      name=f"sheath_{dof}",
    )
    for dof in clamped_dofs
  ]
  bc = _bc.process_boundary_conditions(rod=rod, boundary_conditions_list=sheath)

  vessel = trimesh.load(stl_file, force="mesh")
  vessel.apply_scale(stl_length_scale)
  vessel.apply_translation(-target_xyz[0])
  padding = 3.0 * float(np.max(rod.contact_radii))
  grid_shape = tuple(
    np.ceil((vessel.extents + 2.0 * padding) / grid_spacing).astype(int)
  )
  obstacle = _contact.Obstacle.from_mesh(
    mesh=vessel,
    grid_shape=grid_shape,
    wall_penalty_stiffness=_contact.compute_self_contact_stiffness(
      rod=rod, youngs_modulus=3.0 * shear_modulus, stiffness_ratio=penalty_ratio
    ),
    padding=padding,
    extrap_fill_value=10.0,
    movement=_contact.ObstacleMovement.from_translations(
      jnp.asarray(
        np.einsum("k,d->kd", rod_length - insertion_depths, entry_tangent)
      )
    ),
  )

  problem = _fe.CosseratProblem(
    rod=rod,
    bc=bc,
    nr_solver=_solver.NewtonRaphson(
      solver_type=_solver.NonlinearSolvers.ARMIJO_BACKTRACKING_NEWTON_RAPHSON,
      nonlinear_settings=nonlinear_settings,
      linear_settings=linear_settings,
      verbose=solver_verbose,
    ),
    obstacle=obstacle,
  )
  return ContactSetup(
    rod=rod,
    problem=problem,
    insertion_depths=insertion_depths,
    node_path_coords=node_path_coords,
    target_points=target_curve_fn(np.linspace(0.0, path_length, 400)),
    entry_tangent=entry_tangent,
    origin=target_xyz[0],
  )

### Build and check the vessel before solving

The signed distance along the centerline, evaluated in the vessel body frame, must be
positive and close to the lumen radius. A negative value means the STL is a lumen
surface rather than a wall solid, so the whole lumen would read as inside the
obstacle. A value far from the lumen radius means the STL and the csv disagree in
units or frame.

In [ ]:
setup = build_setup(**baseline_settings)

obstacle = setup.problem.obstacle
contact_radius = float(np.max(setup.rod.contact_radii))
centerline_sdf = np.asarray(
  obstacle.compute_sdf(jnp.asarray(setup.target_points), jnp.zeros(3), jnp.eye(3))
)
spacing = np.asarray(obstacle.spacing)

print(f"insertion steps          {setup.insertion_depths.shape[0]}")
print(f"grid shape               {obstacle.sdf_grid.shape}, "
      f"{obstacle.sdf_grid.size / 1e6:.1f} M cells")
print(f"grid spacing             {np.round(1e3 * spacing, 3)} mm")
print(f"centerline sdf min/mean  {1e3 * centerline_sdf.min():.2f} / "
      f"{1e3 * centerline_sdf.mean():.2f} mm (want > 0, near lumen radius)")
print(f"contact radius           {1e3 * contact_radius:.3f} mm")
print(f"cells across the lumen   {2.0 * centerline_sdf.min() / spacing.max():.1f} "
      f"(want > 10)")
print(f"wall penalty k_w         {obstacle.wall_penalty_stiffness:.3e} N/m^2")

### Solve, post-process, and plotting

In [ ]:
def solve_insertion(
  setup: ContactSetup,
  remnant_mag_flux: jnp.ndarray | None = None,
  applied_mag_flux: jnp.ndarray | None = None,
) -> jnp.ndarray:
  """Solves every insertion step with wall contact and optional magnetic loads.

  Passing both flux arguments as `None` gives the passive catheter, steered by the
  wall alone. Passing both switches on the hard-magnetic couples with no other change.

  Args:
    setup: `ContactSetup` supplying the problem.
    remnant_mag_flux: Array of shape `(num_elems, num_fibers, 3)` [T] of the remnant
      flux in the reference global frame, or `None`.
    applied_mag_flux: Array of shape `(num_steps, 3)` [T] of the applied flux at every
      insertion step, or `None`.

  Returns:
    Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.
  """
  _, disp_history = setup.problem.solve(
    u0=jnp.zeros(setup.rod.num_dofs),
    shear_moduli=jnp.full(
      (setup.rod.fibers.num_elems, setup.rod.fibers.num_fibers), shear_modulus
    ),
    remnant_mag_flux=remnant_mag_flux,
    applied_mag_flux=applied_mag_flux,
  )
  return disp_history


def compute_node_positions(
  rod: _mesher.Rod,
  disp_history: jnp.ndarray,
) -> jnp.ndarray:
  """Computes the current node positions `x_n^k = X_n + u_n^k` at every step.

  Args:
    rod: `Rod` supplying the reference node coordinates `X_n`.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.

  Returns:
    Array of shape `(num_steps, num_nodes, 3)` [m] of node positions.
  """
  num_trans = rod.num_translational_dofs_per_node
  nodal = disp_history.reshape(disp_history.shape[0], rod.num_nodes, -1)
  return rod.node_coords[None, :, :] + nodal[..., :num_trans]


def compute_contact_forces(
  setup: ContactSetup,
  disp_history: jnp.ndarray,
) -> jnp.ndarray:
  """Computes the nodal force exerted by the wall on the rod at every step.

  The obstacle pose of each step is taken from its `ObstacleMovement`. The residual
  assembles the positive gradient of the contact energy as an internal force, so the
  physical force on the rod is its negative. Only translational
  components are returned, since the contact energy depends on the element centre
  alone and its rotational components vanish identically.

  Args:
    setup: `ContactSetup` supplying the rod and the obstacle.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.

  Returns:
    Array of shape `(num_steps, num_nodes, 3)` [N] of nodal contact forces.
  """
  rod = setup.rod
  obstacle = setup.problem.obstacle
  num_trans = rod.num_translational_dofs_per_node

  def step_forces(disp: jnp.ndarray, step_idx: jnp.ndarray) -> jnp.ndarray:
    """Evaluates the nodal contact force of one converged state at its pose."""
    obj_trans, obj_rot_mat = obstacle.pose_at_step(step_idx)
    internal, _ = _contact.compute_wall_contact_forces_and_tangent(
      u_elems=disp[rod.elem_dof_mat],
      rod=rod,
      obstacle=obstacle,
      compute_jacobian=False,
      obj_trans=obj_trans,
      obj_rot_mat=obj_rot_mat,
    )
    return -internal.reshape(rod.num_nodes, -1)[:, :num_trans]

  return jax.vmap(step_forces)(disp_history, jnp.arange(disp_history.shape[0]))


def compute_penetration_ratio(
  setup: ContactSetup,
  disp_history: jnp.ndarray,
) -> jnp.ndarray:
  """Computes the element overlap `ε / r = (r − φ(x_c)) / r` at every step.

  The contact energy is evaluated at the deformed element centre `x_c`, the mean of
  the two node positions, against the vessel at the pose of that step, so the same
  point and pose are used here. Positive values are
  penetrations into the wall, and values below `−1` mean the element centre is more
  than one radius clear of it.

  Args:
    setup: `ContactSetup` supplying the rod and the obstacle.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.

  Returns:
    Array of shape `(num_steps, num_elems)` of overlap ratios.
  """
  obstacle = setup.problem.obstacle
  positions = compute_node_positions(rod=setup.rod, disp_history=disp_history)
  centers = 0.5 * (positions[:, :-1] + positions[:, 1:])
  sdf = jax.vmap(obstacle.compute_sdf)(
    centers, obstacle.movement.translations, obstacle.movement.rotation_matrices
  )
  radii = setup.rod.contact_radii[None, :]
  return (radii - sdf) / radii


def plot_insertion(
  setup: ContactSetup,
  disp_history: np.ndarray,
  num_snapshots: int = 5,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the inserted catheter at evenly spaced depths in the vessel body frame.

  The vessel only translates, so the rod is mapped into its body frame by
  `x_body = x_lab − T_k`, which draws the vessel fixed and the rod advancing. Only
  nodes inside the vessel, `σ ≥ 0`, are drawn. The axis limits and the box aspect are
  both set from everything drawn, the centerline and the rod, with every axis held
  open to at least a tenth of the longest, so all three axes share one scale.

  Args:
    setup: `ContactSetup` supplying the rod, the path coordinates, and the centerline.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.
    num_snapshots: Number of insertion steps drawn, positive.

  Returns:
    The figure and its single 3D axis.
  """
  positions = 1e3 * (
    np.asarray(
      compute_node_positions(rod=setup.rod, disp_history=jnp.asarray(disp_history))
    )
    - np.asarray(setup.problem.obstacle.movement.translations)[:, None, :]
  )
  steps = np.linspace(0, setup.insertion_depths.shape[0] - 1, num_snapshots)
  colors = plt.cm.viridis(np.linspace(0.0, 1.0, num_snapshots))

  centerline = 1e3 * setup.target_points
  drawn = [centerline]

  fig = plt.figure(figsize=(6.0, 5.0))
  ax = fig.add_subplot(projection="3d")
  ax.plot(*centerline.T, color="k", ls="dashed", lw=1.0, label="Vessel centerline")
  for color, step in zip(colors, steps.round().astype(int)):
    free = setup.node_path_coords[step] >= 0.0
    drawn.append(positions[step, free])
    ax.plot(*positions[step, free].T, color=color, lw=1.5,
            label=f"$d$ = {1e3 * setup.insertion_depths[step]:.1f} mm")
  points = np.concatenate(drawn)
  center = 0.5 * (points.min(axis=0) + points.max(axis=0))
  span = np.ptp(points, axis=0)
  span = np.maximum(span, 0.1 * span.max())
  ax.set_xlim(center[0] - 0.5 * span[0], center[0] + 0.5 * span[0])
  ax.set_ylim(center[1] - 0.5 * span[1], center[1] + 0.5 * span[1])
  ax.set_zlim(center[2] - 0.5 * span[2], center[2] + 0.5 * span[2])
  ax.set_box_aspect(span)
  ax.set_xlabel(r"$x$ [mm]")
  ax.set_ylabel(r"$y$ [mm]")
  ax.set_zlabel(r"$z$ [mm]")
  ax.legend(fontsize=7, loc="upper left")
  return fig, ax


def plot_contact_line_load(
  setup: ContactSetup,
  contact_forces: np.ndarray,
  num_snapshots: int = 5,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the wall line load along the path coordinate at evenly spaced depths.

  The nodal force is divided by the mean element length, which turns it into the line
  load `|f|` [N/m] the penalty model applies along the wall.

  Args:
    setup: `ContactSetup` supplying the rod, the depths, and the path coordinates.
    contact_forces: Array of shape `(num_steps, num_nodes, 3)` [N] of nodal forces.
    num_snapshots: Number of insertion steps drawn, positive.

  Returns:
    The figure and its single axis.
  """
  line_load = np.linalg.norm(contact_forces, axis=-1) / float(
    np.mean(setup.rod.elem_arc_lengths)
  )
  steps = np.linspace(0, setup.insertion_depths.shape[0] - 1, num_snapshots)
  colors = plt.cm.viridis(np.linspace(0.0, 1.0, num_snapshots))

  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  for color, step in zip(colors, steps.round().astype(int)):
    free = setup.node_path_coords[step] >= 0.0
    ax.plot(1e3 * setup.node_path_coords[step, free], line_load[step, free],
            color=color, lw=1.2,
            label=f"$d$ = {1e3 * setup.insertion_depths[step]:.1f} mm")
  ax.set_xlabel(r"Path coordinate $\sigma$ [mm]")
  ax.set_ylabel(r"Wall line load $|f|$ [N/m]")
  ax.legend(fontsize=7)
  return fig, ax


def plot_penetration(
  setup: ContactSetup,
  penetration_ratio: np.ndarray,
  tolerance: float = 0.05,
) -> tuple[plt.Figure, plt.Axes]:
  """Plots the largest overlap ratio `max_e ε / r` against the insertion depth.

  Args:
    setup: `ContactSetup` supplying the depths.
    penetration_ratio: Array of shape `(num_steps, num_elems)` of overlap ratios.
    tolerance: Acceptable overlap ratio drawn as a reference line, in `(0, 1)`.

  Returns:
    The figure and its single axis.
  """
  fig, ax = plt.subplots(figsize=(5.0, 3.5))
  ax.plot(1e3 * setup.insertion_depths, penetration_ratio.max(axis=1), color="k",
          lw=1.2)
  ax.axhline(tolerance, color="r", ls="dotted", lw=1.0)
  ax.set_xlabel(r"Insertion depth $d$ [mm]")
  ax.set_ylabel(r"Largest overlap $\varepsilon / r$")
  return fig, ax

### Passive insertion

In [ ]:
start = time.perf_counter()
solve_passive = jax.jit(ft.partial(solve_insertion, setup=setup))
disp_history = solve_passive()
print(f"solve time {time.perf_counter() - start:.1f} s (includes compilation)")

contact_forces = np.asarray(
  compute_contact_forces(setup=setup, disp_history=disp_history)
)
penetration_ratio = np.asarray(
  compute_penetration_ratio(setup=setup, disp_history=disp_history)
)
print(f"largest overlap eps/r    {penetration_ratio.max():.3%}")
peak_force = np.linalg.norm(contact_forces, axis=-1).max()
print(f"largest nodal force      {1e3 * peak_force:.3f} mN")
np.savez(
  results_dir / "passive.npz",
  disp=np.asarray(disp_history),
  contact_forces=contact_forces,
  penetration_ratio=penetration_ratio,
  **baseline_settings,
)

### Results

In [ ]:
figures = {
  "insertion": plot_insertion(setup=setup, disp_history=np.asarray(disp_history)),
  "line_load": plot_contact_line_load(setup=setup, contact_forces=contact_forces),
  "penetration": plot_penetration(
    setup=setup,
    penetration_ratio=penetration_ratio,
    tolerance=0.13,
  ),
}
for name, (fig, _) in figures.items():
  fig.savefig(results_dir / f"passive_{name}.pdf", bbox_inches="tight")
plt.show()

### Insertion animation

The vessel is drawn fixed in its body frame and the rod is mapped into it, so the gif
reads as an insertion. Only every `frame_interval`-th step and the last step are
rendered, since each frame is a Polyscope screenshot. The camera looks along the normal
of the best-fit plane of the centerline, with the up axis set to the Cartesian axis
closest to its longest principal direction.

In [ ]:
def render_insertion_gif(
  setup: ContactSetup,
  disp_history: np.ndarray,
  contact_forces: np.ndarray,
  gif_path: pathlib.Path,
  frame_interval: int = 15,
) -> None:
  """Renders the insertion with Polyscope, colored by wall force, and writes a gif.

  Algorithm.
    1. Register the vessel at the identity pose, which is its body frame, the rod as a
       tube of its contact radius colored by the element force magnitude on a range
       fixed over all steps, and the csv centerline as a thin black curve that must
       run down the middle of the lumen.
    2. Center the camera on the centerline centroid `c`. The right singular vectors
       `v₁, v₂, v₃` of the centered centerline give its longest direction `v₁` and its
       best-fit plane normal `v₃`. Set the up axis to the Cartesian axis closest to
       `v₁`, and place the camera at `c + 1.6 s v₃`, where `s` is the largest extent.
    3. For every `frame_interval`-th step and the last step, map the node positions
       into the body frame by `x_body = Rᵀ (x_lab − T_k)`, average the nodal force
       magnitude onto elements, update the rod, and save a screenshot.
    4. Assemble the screenshots into a gif.

  Args:
    setup: `ContactSetup` supplying the rod, the moving obstacle, and the centerline.
    disp_history: Array of shape `(num_steps, num_dofs)` [m, rad] of converged states.
    contact_forces: Array of shape `(num_steps, num_nodes, 3)` [N] of nodal forces.
    gif_path: Output path of the gif. Frames are written to a sibling folder.
    frame_interval: Number of load steps between rendered frames, positive,
      typically 5 to 10.
  """
  rod = setup.rod
  obstacle = setup.problem.obstacle
  num_steps = disp_history.shape[0]
  positions = np.asarray(
    compute_node_positions(rod=rod, disp_history=jnp.asarray(disp_history))
  )
  force_magnitude = np.linalg.norm(contact_forces, axis=-1)
  contact_radii = np.asarray(rod.contact_radii)
  node_radii = np.concatenate(
    [
      contact_radii[:1],
      0.5 * (contact_radii[:-1] + contact_radii[1:]),
      contact_radii[-1:],
    ]
  )

  _viz.init_scene_ps()
  ps.set_window_size(1920, 1440)  # screenshot resolution in pixels
  # ps.set_SSAA_factor(2)           # supersampling: smoother edges, same pixel size
  _viz.plot_obstacle_ps(
    dataclasses.replace(obstacle, movement=None), step_idx=0, name="vessel"
  )
  _viz.plot_rod_centerline_ps(
    np.zeros((rod.num_nodes, 3)),
    node_radii=node_radii,
    elem_field=np.zeros(rod.fibers.num_elems),
    elem_field_val_range=(0.0, float(force_magnitude.max())),
    cmap="viridis",
    name="rod",
  )
  # The csv centerline must run down the middle of the STL lumen. If it does not, the
  # two files describe different geometry or frames.
  ps.register_curve_network(
    "centerline", setup.target_points, edges="line", color=(0.0, 0.0, 0.0)
  ).set_radius(0.3 * float(np.max(contact_radii)), relative=False)

  center = setup.target_points.mean(axis=0)
  _, _, principal = np.linalg.svd(setup.target_points - center)
  span = float(np.ptp(setup.target_points, axis=0).max())
  ps.set_up_dir(("x_up", "y_up", "z_up")[int(np.argmax(np.abs(principal[0])))])
  ps.look_at(
    camera_location=tuple(center + 1.6 * span * principal[2]), target=tuple(center)
  )

  frame_dir = gif_path.with_suffix("")
  frame_dir.mkdir(parents=True, exist_ok=True)
  frame_steps = sorted(set(range(0, num_steps, frame_interval)) | {num_steps - 1})
  for step in frame_steps:
    obj_trans, obj_rot_mat = obstacle.pose_at_step(step)
    _viz.update_rod_centerline_ps(
      np.asarray(
        _utils.affine_transform(
          positions[step],
          np.asarray(obj_trans),
          np.asarray(obj_rot_mat),
          "GLOBAL_TO_LOCAL",
        )
      ),
      elem_field=0.5 * (force_magnitude[step, :-1] + force_magnitude[step, 1:]),
      name="rod",
    )
    ps.screenshot(str(frame_dir / f"frame_{step:04d}.png"), transparent_bg=False)

  imageio.mimsave(
    gif_path,
    [imageio.imread(frame_dir / f"frame_{step:04d}.png") for step in frame_steps],
  )


render_insertion_gif(
  setup=setup,
  disp_history=np.asarray(disp_history),
  contact_forces=contact_forces,
  gif_path=results_dir / "passive_insertion.gif",
  frame_interval=5,
)